# Grafo de Sensores de Tráfico — Madrid

Construye un grafo dirigido de los sensores de tráfico de Madrid anclados
a la red vial real descargada desde OpenStreetMap con **OSMnx**.

## Índice

| # | Sección | Descripción |
|---|---------|-------------|
| 1 | Imports y parámetros | Librerías, rutas y constantes globales |
| 2 | Carga de sensores | Metadatos desde el CSV de tráfico |
| 3 | GeoDataFrame de sensores | Geometría `Point`, CRS WGS-84, distribución geográfica |
| 4 | Red vial OSM | Descarga de la red vial con OSMnx |
| 5 | Sensores sobre la red vial | Matplotlib (calles + puntos) y Folium interactivo |
| 6 | Map Matching — Anclaje a red vial | Snap con diagnóstico de calidad, visualización y CSV |
| 7 | Distancias reales | BallTree + Dijkstra sobre la red OSM |
| 8 | Construcción del grafo | MultiDiGraph con atributos de sensor |
| 9 | Conectividad máxima | Puentes entre componentes desconectadas |
| 10 | Etiquetado de sentidos | Bearing (rumbo) → ida / vuelta |
| 11 | Visualización estática del grafo | Matplotlib — nodos y aristas sensor-sensor |
| 12 | Visualización interactiva del grafo | Folium — grafo completo con popups |
| 13 | Análisis de conectividad | Estadísticas del grafo sensor |
| 14 | Exportación | GraphML + CSV + PNG + HTML |
| 15 | Búsqueda de rutas (Dijkstra) | Ruta óptima entre sensores: tramos, distancias y visualización |
| 16 | Búsqueda por coordenadas GPS | Snap lat/lon → sensor más cercano (BallTree) → Dijkstra |
| 17 | Routing OSM real + sensores en ruta | Dijkstra sobre la red vial real, sensores como observaciones |

**Dependencias:**
```
pip install osmnx networkx scikit-learn pandas numpy matplotlib folium shapely tqdm
```

## 1. Imports y parámetros

In [ ]:
import math
import warnings
from pathlib import Path

import folium
import geopandas as gpd   # pip install geopandas
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
from shapely.geometry import box as shapely_box
from sklearn.neighbors import BallTree

try:
    from tqdm import tqdm
except ImportError:
    def tqdm(it, **kw):
        return it

warnings.filterwarnings("ignore")
ox.settings.log_console = False
ox.settings.use_cache   = True   # cachea descargas → 2ª ejecución instantánea

# ── Rutas ─────────────────────────────────────────────────────────────────────
DATA_PATH  = Path("data/Trafico_MODELOS2.csv")
OUTPUT_DIR = Path("outputs/v1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Parámetros del grafo ──────────────────────────────────────────────────────
MIN_DIST_M = 30          # distancia mínima entre sensores; por debajo se asume carril contrario
K_VECINOS  = 3           # vecinos dirigidos válidos por sensor
K_QUERY    = K_VECINOS + 15   # candidatos extra para absorber el filtro MIN_DIST
R_EARTH    = 6_371_000   # radio de la Tierra en metros
OSM_MARGIN = 0.015       # ~1.5 km de margen alrededor del bounding box de sensores

# ── Paleta de colores ─────────────────────────────────────────────────────────
COL_A      = "#1565C0"   # azul oscuro  → sentido A (calzada de ida)
COL_B      = "#B71C1C"   # rojo oscuro  → sentido B (calzada de vuelta)
COL_UNICO  = "#E65100"   # naranja      → sin pareja, sentido único
COL_EDGE   = "#78909C"   # gris azulado → aristas en el mapa Folium
COL_ARROW  = "#37474F"   # gris oscuro  → flechas de dirección

print("Librerías cargadas correctamente")
print(f"  OSMnx   {ox.__version__}")
print(f"  NetworkX {nx.__version__}")

## 2. Carga de metadatos de sensores

El CSV tiene 37.5 M de filas y ~9 GB. Solo se necesitan las 5 columnas de
metadatos, que son constantes para cada sensor.
Se lee por chunks y se para en cuanto se tienen todos los sensores únicos.

In [ ]:
META_COLS = ["id", "nombre", "longitud", "latitud", "tipo_elem"]
frames, seen = [], set()

for chunk in pd.read_csv(DATA_PATH, usecols=META_COLS,
                         chunksize=500_000, low_memory=False):
    nuevos = chunk[~chunk["id"].isin(seen)].drop_duplicates("id")
    if len(nuevos):
        frames.append(nuevos)
        seen.update(nuevos["id"].tolist())
    if len(seen) >= 1069:   # número total de sensores únicos conocido
        break

sensors = (
    pd.concat(frames, ignore_index=True)
    .drop_duplicates("id")
    .dropna(subset=["latitud", "longitud"])
    .reset_index(drop=True)
)
sensors["latitud"]  = pd.to_numeric(sensors["latitud"],  errors="coerce")
sensors["longitud"] = pd.to_numeric(sensors["longitud"], errors="coerce")
sensors = sensors.dropna(subset=["latitud", "longitud"]).reset_index(drop=True)

print(f"Sensores cargados : {len(sensors)}")
print(f"\nDistribución por tipo:")
print(sensors["tipo_elem"].value_counts().to_string())
sensors[["id", "nombre", "latitud", "longitud", "tipo_elem"]].head()

## 3. GeoDataFrame de sensores

Cada sensor se convierte en un **punto geográfico** con `geopandas`.
La columna `id` del CSV se conserva explícitamente como `sensor_id` en el GeoDataFrame.

- **CRS `EPSG:4326`** (WGS-84): sistema de referencia GPS estándar, grados decimales.
- `gpd.points_from_xy(longitud, latitud)` crea geometrías `Point` compatibles con GIS.
- No se elimina ningún sensor aunque esté en la periferia o con pocas conexiones.

In [ ]:
# ── GeoDataFrame: coordenadas → geometría Point ──────────────────────────────
# La columna `id` del CSV es el identificador único del sensor en todo el sistema.
# Se conserva sin modificar para que todas las celdas posteriores sigan funcionando.
geometry = gpd.points_from_xy(sensors["longitud"], sensors["latitud"])

gdf = gpd.GeoDataFrame(
    sensors.copy(),
    geometry=geometry,
    crs="EPSG:4326",      # WGS-84: sistema de referencia GPS estándar
)
gdf = gdf.rename(columns={"id": "sensor_id"})   # nombre explícito del id único

print("GeoDataFrame creado:")
print(f"  Registros : {len(gdf)}")
print(f"  CRS       : {gdf.crs}")
print(f"  Columnas  : {list(gdf.columns)}")
print()
bbox = gdf.total_bounds   # (minx, miny, maxx, maxy) en grados decimales
print("Extensión geográfica:")
print(f"  Longitud  : {bbox[0]:.4f}°E  →  {bbox[2]:.4f}°E")
print(f"  Latitud   : {bbox[1]:.4f}°N  →  {bbox[3]:.4f}°N")
print()
print("Distribución por tipo:")
print(gdf["tipo_elem"].value_counts().to_string())
print()

# ── Visualización previa: solo sensores, sin red vial ────────────────────────
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_facecolor("#1a1a2e")
fig.patch.set_facecolor("#1a1a2e")

_COL_TIPO = {"URB": "#FF6F00", "M30": "#FFD600"}

for tipo, grp in gdf.groupby("tipo_elem"):
    ax.scatter(
        grp["longitud"], grp["latitud"],
        s=8 if tipo == "URB" else 25,
        c=_COL_TIPO.get(tipo, "#FFFFFF"),
        alpha=0.75, zorder=3,
        label=f"{tipo}  ({len(grp)} sensores)",
    )

ax.set_aspect("equal")
ax.axis("off")
ax.legend(facecolor="#2d2d44", edgecolor="white", labelcolor="white", fontsize=11)
ax.set_title(
    f"Distribución geográfica — {len(gdf)} sensores (sin red vial)",
    color="white", fontsize=12, pad=10,
)
plt.tight_layout()
plt.show()
print("La red vial como fondo se añade en §5.")
print()
gdf[["sensor_id", "nombre", "latitud", "longitud", "tipo_elem", "geometry"]].head()

## 4. Red vial OSM — descarga con OSMnx

`graph_from_polygon` descarga **solo las carreteras** (network_type='drive')
dentro del bounding box de los sensores más un margen de seguridad.

- `retain_all=True`: conserva nodos sin salida para no romper la topología real.
- `simplify=True`: fusiona nodos intermedios redundantes → grafo más compacto.
- `use_cache=True`: la primera descarga se guarda en disco; la segunda es instantánea.

In [ ]:
north = sensors["latitud"].max()  + OSM_MARGIN
south = sensors["latitud"].min()  - OSM_MARGIN
east  = sensors["longitud"].max() + OSM_MARGIN
west  = sensors["longitud"].min() - OSM_MARGIN

print(f"Bounding box:")
print(f"  Latitud  : {south:.4f}° N  →  {north:.4f}° N")
print(f"  Longitud : {west:.4f}° E  →  {east:.4f}° E")
print("\nDescargando red vial (1-2 min la primera vez; se cachea automáticamente)...")

area  = shapely_box(west, south, east, north)   # (minx, miny, maxx, maxy)
G_osm = ox.graph_from_polygon(
    area,
    network_type="drive",
    retain_all=True,
    simplify=True,
)

print(f"\nRed OSM descargada:")
print(f"  Nodos   : {len(G_osm.nodes):,}")
print(f"  Aristas : {len(G_osm.edges):,}")

## 5. Sensores sobre la red vial

Los sensores del GeoDataFrame se superponen sobre la **red vial real** de Madrid.

### Matplotlib
`ox.plot_graph()` dibuja las calles de OSMnx; `ax.scatter()` añade los sensores encima.
Color por tipo: **naranja** = URB (urbano) · **amarillo** = M30 (autovía).

### Folium
Mapa interactivo sobre teselas CartoDB. Cada marcador incluye el `sensor_id` en el tooltip
y un popup con nombre, tipo y coordenadas exactas.

In [ ]:
# ── Matplotlib: red vial OSM como fondo + sensores encima ────────────────────
# ox.plot_graph devuelve (fig, ax) con las calles ya dibujadas en coordenadas lon/lat.
# Añadimos los sensores encima con ax.scatter() usando sus coordenadas geográficas.
fig, ax = ox.plot_graph(
    G_osm,
    figsize=(16, 16),
    bgcolor="#1a1a2e",
    node_size=0,          # ocultar nodos OSM — son de la red, no son sensores
    edge_color="#546E7A",
    edge_linewidth=0.35,
    edge_alpha=0.65,
    show=False,
    close=False,
)

_COL_TIPO_MPL = {"URB": "#FF6F00", "M30": "#FFD600"}

for tipo, grp in gdf.groupby("tipo_elem"):
    ax.scatter(
        grp["longitud"], grp["latitud"],
        s=10 if tipo == "URB" else 30,
        c=_COL_TIPO_MPL.get(tipo, "#FFFFFF"),
        zorder=5, alpha=0.85,
        label=f"{tipo}  ({len(grp)} sensores)",
    )

ax.legend(loc="lower right", facecolor="#2d2d44", edgecolor="white",
          labelcolor="white", fontsize=11)
ax.set_title(
    f"Sensores de tráfico sobre la red vial — Madrid"
    f"  |  {len(gdf)} sensores"
    f"  ·  red OSM: {len(G_osm.nodes):,} nodos / {len(G_osm.edges):,} aristas",
    color="white", fontsize=12, pad=14,
)
fig.savefig(OUTPUT_DIR / "sensores_sobre_red_vial.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Figura guardada: {OUTPUT_DIR}/sensores_sobre_red_vial.png")
print()

# ── Folium: sensores interactivos sobre mapa real (sin aristas del grafo) ────
# Cada CircleMarker = un sensor físico, coloreado por tipo_elem.
# El sensor_id se conserva en el tooltip y en el popup.
_lat_c = gdf["latitud"].mean()
_lon_c = gdf["longitud"].mean()

m_sensores = folium.Map(
    location=[_lat_c, _lon_c], zoom_start=13, tiles="CartoDB positron"
)

_COL_TIPO_FOL = {"URB": "#E65100", "M30": "#6A1B9A"}

for _, row in gdf.iterrows():
    color = _COL_TIPO_FOL.get(row["tipo_elem"], "#607D8B")
    folium.CircleMarker(
        location=[row["latitud"], row["longitud"]],
        radius=5 if row["tipo_elem"] == "URB" else 7,
        color=color, fill=True, fill_color=color, fill_opacity=0.85,
        tooltip=f"sensor_id: {row['sensor_id']}",
        popup=folium.Popup(
            (
                f"<b>sensor_id: {row['sensor_id']}</b><br>"
                f"{row['nombre']}<br>"
                f"Tipo: {row['tipo_elem']}<br>"
                f"({row['latitud']:.5f}&deg;N,&nbsp;{row['longitud']:.5f}&deg;E)"
            ),
            max_width=240,
        ),
    ).add_to(m_sensores)

_legend_s = (
    "<div style='position:fixed;bottom:30px;left:30px;z-index:1000;"
    "background:white;padding:12px 16px;border-radius:8px;"
    "border:1px solid #ccc;font-size:13px;line-height:1.8;'>"
    "<b>Sensores de tr&aacute;fico &mdash; ubicaci&oacute;n real</b><br>"
    "<span style='color:#E65100;font-size:16px;'>&#9679;</span>"
    " URB &mdash; Urbano<br>"
    "<span style='color:#6A1B9A;font-size:16px;'>&#9679;</span>"
    " M30 &mdash; Autov&iacute;a<br>"
    "<small>Clic en sensor &rarr; popup con sensor_id</small>"
    "</div>"
)
m_sensores.get_root().html.add_child(folium.Element(_legend_s))

out_sensores = OUTPUT_DIR / "sensores_madrid.html"
m_sensores.save(str(out_sensores))
print(f"Mapa interactivo guardado: {out_sensores}")
m_sensores

## 6. Map Matching — Anclaje de sensores a la red vial

**Map matching** asocia cada sensor (coordenadas GPS) con el nodo de la red vial
más cercano. Es el primer paso para calcular distancias reales entre sensores.

### Algoritmo
1. `ox.nearest_nodes(G_osm, X=lons, Y=lats)` → nodo OSM más próximo por sensor.
2. Se calcula la **distancia de snap** (haversine sensor ↔ nodo OSM asignado).
3. Se etiqueta la calidad: `bueno` < 50 m · `aceptable` 50–200 m · `revisar` > 200 m.
4. Se construye la tabla `sensor_id → osm_node` para uso en las secciones siguientes.

### Por qué puede haber snap elevado
- Sensor en zona sin red OSM (parque, zona peatonal, vial privado).
- La posición GPS del sensor no coincide exactamente con el carril.
- El nodo más próximo en el grafo simplificado puede estar a cierta distancia.

> Ningún sensor se elimina por tener snap elevado; todos permanecen en el grafo.

In [ ]:
import math

def _haversine_m(lat1, lon1, lat2, lon2, R=6_371_000.0):
    """Distancia haversine en metros entre dos puntos GPS."""
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlon / 2) ** 2
    return R * 2 * math.asin(math.sqrt(a))


# ── Paso 1: snap GPS → nodo OSM más cercano ──────────────────────────────────
# ox.nearest_nodes busca el nodo de la red vial más próximo en coordenadas
# proyectadas. Para grafos grandes es mucho más rápido que calcular todas
# las distancias haversine a mano.
snapped_ids = ox.nearest_nodes(
    G_osm,
    X=sensors["longitud"].tolist(),   # X = longitud
    Y=sensors["latitud"].tolist(),    # Y = latitud
)
sensors["osm_node"] = [int(n) for n in snapped_ids]

# ── Paso 2: distancia de snap para cada sensor ────────────────────────────────
# OSMnx almacena coordenadas como  node['x'] = longitud, node['y'] = latitud.
snap_dist_m = []
for i, (_, row) in enumerate(sensors.iterrows()):
    nd = G_osm.nodes[int(snapped_ids[i])]
    snap_dist_m.append(
        _haversine_m(
            float(row["latitud"]),  float(row["longitud"]),
            float(nd["y"]),         float(nd["x"]),
        )
    )
sensors["snap_dist_m"] = snap_dist_m

# ── Paso 3: tabla de mapping  sensor_id → nodo vial ─────────────────────────
snap_map = pd.DataFrame({
    "sensor_id":   sensors["id"],
    "tipo_elem":   sensors["tipo_elem"],
    "osm_node":    sensors["osm_node"],
    "snap_dist_m": [round(d, 1) for d in snap_dist_m],
    "lat_sensor":  sensors["latitud"].round(6),
    "lon_sensor":  sensors["longitud"].round(6),
    "lat_nodo":    [round(float(G_osm.nodes[n]["y"]), 6) for n in sensors["osm_node"]],
    "lon_nodo":    [round(float(G_osm.nodes[n]["x"]), 6) for n in sensors["osm_node"]],
}).reset_index(drop=True)

# ── Paso 4: etiqueta de calidad ───────────────────────────────────────────────
def _calidad(d):
    if d < 50:    return "bueno"      # < 50 m: sensor sobre la calzada
    if d < 200:   return "aceptable"  # 50-200 m: offset razonable
    return "revisar"                  # > 200 m: posible error de posición

snap_map["calidad"] = snap_map["snap_dist_m"].map(_calidad)
sensors["snap_calidad"] = snap_map["calidad"].values

# ── Estadísticas ──────────────────────────────────────────────────────────────
print("=" * 52)
print("  MAP MATCHING — Diagnóstico de calidad")
print("=" * 52)
print(f"  Sensores anclados         : {len(sensors)}")
print(f"  Nodos OSM únicos          : {sensors['osm_node'].nunique()}")
print(f"  Colisiones de snap        : {len(sensors) - sensors['osm_node'].nunique()}")
print()
print("  Distancia de snap (m):")
print(f"    Media                   : {snap_map['snap_dist_m'].mean():.1f}")
print(f"    Mediana                 : {snap_map['snap_dist_m'].median():.1f}")
print(f"    Máxima                  : {snap_map['snap_dist_m'].max():.1f}")
print(f"    Mínima                  : {snap_map['snap_dist_m'].min():.1f}")
print(f"    Desv. estándar          : {snap_map['snap_dist_m'].std():.1f}")
print()
print("  Calidad del matching:")
for cal in ["bueno", "aceptable", "revisar"]:
    cnt = (snap_map["calidad"] == cal).sum()
    pct = cnt / len(snap_map) * 100
    bar = "█" * int(pct / 2)
    print(f"    {cal:<10s}: {cnt:>4d}  ({pct:5.1f}%)  {bar}")
print("=" * 52)
print()
print("Sensores con mayor snap (potencialmente fuera de red vial):")
snap_map.sort_values("snap_dist_m", ascending=False).head(10)

In [ ]:
# ── Matplotlib: diagnóstico visual del map matching ──────────────────────────
_COL_CAL = {"bueno": "#4CAF50", "aceptable": "#FFC107", "revisar": "#F44336"}

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.patch.set_facecolor("#1a1a2e")

# ── Panel izquierdo: histograma de distancias de snap ────────────────────────
ax_hist = axes[0]
ax_hist.set_facecolor("#1a1a2e")

for cal in ["bueno", "aceptable", "revisar"]:
    mask = snap_map["calidad"] == cal
    vals = snap_map.loc[mask, "snap_dist_m"]
    if vals.empty:
        continue
    ax_hist.hist(vals, bins=30, color=_COL_CAL[cal], alpha=0.80,
                 label=f"{cal}  (n={mask.sum()})")

ax_hist.axvline(50,  color="#FFC107", linestyle="--", linewidth=1, alpha=0.7)
ax_hist.axvline(200, color="#F44336", linestyle="--", linewidth=1, alpha=0.7)
ax_hist.set_xlabel("Distancia de snap (m)", color="white")
ax_hist.set_ylabel("Número de sensores",    color="white")
ax_hist.set_title("Distribución de distancias de snap", color="white", fontsize=12)
ax_hist.legend(facecolor="#2d2d44", edgecolor="white", labelcolor="white")
ax_hist.tick_params(colors="white")
for spine in ax_hist.spines.values():
    spine.set_color("#555")

# ── Panel derecho: mapa de matching sobre la red vial ────────────────────────
ax_map = axes[1]
ax_map.set_facecolor("#1a1a2e")

# Red vial OSM como fondo muy tenue
for u, v, data in G_osm.edges(data=True):
    xu, yu = G_osm.nodes[u]["x"], G_osm.nodes[u]["y"]
    xv, yv = G_osm.nodes[v]["x"], G_osm.nodes[v]["y"]
    ax_map.plot([xu, xv], [yu, yv], color="#37474F", linewidth=0.3,
                alpha=0.4, zorder=1)

# Líneas sensor → nodo OSM asignado (coloreadas por calidad)
for _, r in snap_map.iterrows():
    ax_map.plot(
        [r["lon_sensor"], r["lon_nodo"]],
        [r["lat_sensor"], r["lat_nodo"]],
        color=_COL_CAL[r["calidad"]], linewidth=0.8, alpha=0.6, zorder=2,
    )

# Nodos OSM asignados
ax_map.scatter(
    snap_map["lon_nodo"], snap_map["lat_nodo"],
    s=6, c="#00BCD4", zorder=3, alpha=0.6, label="Nodo OSM asignado",
)

# Sensores (coloreados por calidad)
for cal, grp in snap_map.groupby("calidad"):
    ax_map.scatter(
        grp["lon_sensor"], grp["lat_sensor"],
        s=12, c=_COL_CAL[cal], zorder=4, alpha=0.85, label=f"Sensor — {cal}",
    )

ax_map.set_aspect("equal")
ax_map.axis("off")
ax_map.legend(facecolor="#2d2d44", edgecolor="white", labelcolor="white",
              fontsize=10, loc="lower right")
ax_map.set_title(
    "Map matching: sensor (●) → nodo OSM (◆)  |  línea = distancia de snap",
    color="white", fontsize=11,
)

plt.suptitle(
    f"Map Matching — {len(snap_map)} sensores  "
    f"|  snap medio: {snap_map['snap_dist_m'].mean():.1f} m  "
    f"|  máximo: {snap_map['snap_dist_m'].max():.1f} m",
    color="white", fontsize=13, y=1.01,
)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "map_matching.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Figura guardada: {OUTPUT_DIR}/map_matching.png")

In [ ]:
# ── Folium: map matching interactivo ─────────────────────────────────────────
# Cada sensor muestra: sensor_id, osm_node, snap_dist_m, calidad.
# La línea conecta el sensor con su nodo vial asignado.
_lat_c = sensors["latitud"].mean()
_lon_c = sensors["longitud"].mean()

m_mm = folium.Map(location=[_lat_c, _lon_c], zoom_start=13,
                  tiles="CartoDB positron")

_COL_CAL_F = {"bueno": "#4CAF50", "aceptable": "#FFC107", "revisar": "#F44336"}

for _, r in snap_map.iterrows():
    color = _COL_CAL_F[r["calidad"]]

    # Línea sensor → nodo OSM
    folium.PolyLine(
        [(r["lat_sensor"], r["lon_sensor"]),
         (r["lat_nodo"],   r["lon_nodo"])],
        color=color, weight=1.5, opacity=0.55,
    ).add_to(m_mm)

    # Nodo OSM asignado
    folium.CircleMarker(
        location=[r["lat_nodo"], r["lon_nodo"]],
        radius=3, color="#00BCD4",
        fill=True, fill_color="#00BCD4", fill_opacity=0.75,
        tooltip=f"OSM: {r['osm_node']}",
    ).add_to(m_mm)

    # Sensor
    folium.CircleMarker(
        location=[r["lat_sensor"], r["lon_sensor"]],
        radius=5, color=color,
        fill=True, fill_color=color, fill_opacity=0.85,
        tooltip=f"sensor_id: {r['sensor_id']}  ({r['snap_dist_m']:.0f} m)",
        popup=folium.Popup(
            (
                f"<b>sensor_id: {r['sensor_id']}</b><br>"
                f"Tipo: {r['tipo_elem']}<br>"
                f"OSM node: {r['osm_node']}<br>"
                f"Snap: <b>{r['snap_dist_m']:.1f} m</b><br>"
                f"Calidad: <b style='color:{color};'>{r['calidad']}</b>"
            ),
            max_width=220,
        ),
    ).add_to(m_mm)

_legend_mm = (
    "<div style='position:fixed;bottom:30px;left:30px;z-index:1000;"
    "background:white;padding:12px 16px;border-radius:8px;"
    "border:1px solid #ccc;font-size:13px;line-height:1.9;'>"
    "<b>Map Matching &mdash; calidad del snap</b><br>"
    "<span style='color:#4CAF50;font-size:15px;'>&#9679;</span>"
    " Bueno &lt; 50 m<br>"
    "<span style='color:#FFC107;font-size:15px;'>&#9679;</span>"
    " Aceptable 50&ndash;200 m<br>"
    "<span style='color:#F44336;font-size:15px;'>&#9679;</span>"
    " Revisar &gt; 200 m<br>"
    "<span style='color:#00BCD4;font-size:15px;'>&#9679;</span>"
    " Nodo OSM asignado<br>"
    "<small>Clic en sensor &rarr; popup con sensor_id y distancia</small>"
    "</div>"
)
m_mm.get_root().html.add_child(folium.Element(_legend_mm))

out_mm = OUTPUT_DIR / "map_matching.html"
m_mm.save(str(out_mm))
print(f"Mapa interactivo guardado: {out_mm}")

# ── Exportar tabla de mapping como CSV ───────────────────────────────────────
# sensor_id → osm_node + diagnóstico de calidad.
# Este fichero es la referencia espacial del sistema:
#   el modelo de ML puede usarlo para saber qué nodo vial corresponde a cada sensor.
out_csv = OUTPUT_DIR / "sensor_map_matching.csv"
snap_map.to_csv(out_csv, index=False)
print(f"Tabla CSV guardada   : {out_csv}")
print(f"  Columnas: {list(snap_map.columns)}")
print()
m_mm

## 7. Distancias reales por carretera

### Detección de sentidos de circulación

Cada sensor hereda el **sentido de circulación del tramo vial OSM** al que
está anclado (§6 Map Matching). Se calcula la **media circular ponderada**
(peso = longitud del edge) de los bearings de todos los edges salientes
del nodo OSM asignado a ese sensor:

| Bearing medio | Color | Sentido |
|---|---|---|
| 0° – 179.9° | Azul | Sentido A (predominante Norte/Este) |
| 180° – 359.9° | Rojo | Sentido B (predominante Sur/Oeste) |
| Sin edges salientes | Naranja | Indeterminado |

Este criterio no depende de la proximidad entre sensores, sino de la
**geometría real de la red vial**. Dos sensores en el mismo tramo pero
a cualquier distancia reciben el mismo sentido si comparten nodo OSM,
y sentidos distintos si sus nodos OSM tienen bearings opuestos.

### Cálculo de distancias

1. **BallTree haversine** → K candidatos espacialmente más cercanos por sensor.
2. Se recopilan los pares de nodos OSM necesarios.
3. **Dijkstra** en el grafo OSM (`weight='length'`) → distancia real por carretera.

In [ ]:
# ── BallTree sobre coordenadas de los sensores ────────────────────────────────
coords_rad = np.radians(sensors[["latitud", "longitud"]].values)
tree = BallTree(coords_rad, metric="haversine")
dist_rad, idx_nn = tree.query(coords_rad, k=K_QUERY + 1)
dist_m_approx = dist_rad * R_EARTH   # distancias haversine en metros


def compass_bearing(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Rumbo en grados [0, 360) desde (lat1, lon1) hacia (lat2, lon2)."""
    la1, la2 = math.radians(lat1), math.radians(lat2)
    dl = math.radians(lon2 - lon1)
    x  = math.sin(dl) * math.cos(la2)
    y  = math.cos(la1) * math.sin(la2) - math.sin(la1) * math.cos(la2) * math.cos(dl)
    return (math.degrees(math.atan2(x, y)) + 360) % 360


# ── Detección de sentidos por bearing del tramo vial OSM ─────────────────
# Cada sensor hereda la dirección de flujo de los edges salientes de su nodo OSM.
# Media circular ponderada por longitud de edge:
#   bearing ∈ [0°, 180°)   → sentido A — azul
#   bearing ∈ [180°, 360°) → sentido B — rojo
#   sin edges salientes    → indeterminado — naranja

def _mean_bearing_weighted(bearings_deg, weights):
    """Media circular ponderada de ángulos en grados."""
    brads = np.radians(bearings_deg)
    sw    = sum(weights)
    sin_m = sum(w * np.sin(b) for b, w in zip(brads, weights)) / sw
    cos_m = sum(w * np.cos(b) for b, w in zip(brads, weights)) / sw
    return (np.degrees(np.arctan2(sin_m, cos_m)) + 360) % 360


def _sensor_osm_bearing(G_osm, osm_node):
    """Bearing predominante del sensor según edges salientes de su nodo OSM."""
    edges_out = list(G_osm.out_edges(osm_node, data=True))
    if not edges_out:
        return None
    bearings, weights = [], []
    for u, v, data in edges_out:
        nu, nv = G_osm.nodes[u], G_osm.nodes[v]
        b = compass_bearing(float(nu["y"]), float(nu["x"]),
                            float(nv["y"]), float(nv["x"]))
        bearings.append(b)
        weights.append(float(data.get("length", 1.0)))
    return _mean_bearing_weighted(bearings, weights)


sensor_color = {}
for _, row in sensors.iterrows():
    brng = _sensor_osm_bearing(G_osm, int(row["osm_node"]))
    if brng is None:
        sensor_color[row["id"]] = COL_UNICO
    elif brng < 180.0:
        sensor_color[row["id"]] = COL_A
    else:
        sensor_color[row["id"]] = COL_B

n_a = sum(1 for c in sensor_color.values() if c == COL_A)
n_b = sum(1 for c in sensor_color.values() if c == COL_B)
n_u = sum(1 for c in sensor_color.values() if c == COL_UNICO)
print(f"Sentido A — azul   (bearing   0°–180°) : {n_a} sensores")
print(f"Sentido B — rojo   (bearing 180°–360°) : {n_b} sensores")
print(f"Indeterminado — naranja                 : {n_u} sensores")

# ── Recopilar pares OSM para los que necesitamos distancia real ───────────────
needed: dict = {}
for i in range(len(sensors)):
    osm_u = int(sensors.iloc[i]["osm_node"])
    for d_approx, j in zip(dist_m_approx[i][1:], idx_nn[i][1:]):
        if d_approx < MIN_DIST_M:
            continue   # carril contrario → no conectar
        osm_v = int(sensors.iloc[j]["osm_node"])
        if osm_u != osm_v:
            needed.setdefault(osm_u, set()).add(osm_v)

# ── Dijkstra por nodo fuente → cache de distancias por carretera ─────────────
cutoff_m = float(dist_m_approx[:, 1:].max()) * 1.25
osm_dist_cache: dict = {}

print(f"\nCutoff Dijkstra : {cutoff_m:,.0f} m")
print(f"Nodos OSM fuente: {len(needed)}")
print("Calculando distancias reales...")

for osm_u, targets in tqdm(needed.items(), desc="Dijkstra OSM", unit="nodo"):
    try:
        dists = nx.single_source_dijkstra_path_length(
            G_osm, osm_u, cutoff=cutoff_m, weight="length"
        )
    except nx.NodeNotFound:
        dists = {}
    for osm_v in targets:
        osm_dist_cache[(osm_u, osm_v)] = dists.get(osm_v)

n_alc = sum(1 for v in osm_dist_cache.values() if v is not None)
print(f"\nPares calculados : {len(osm_dist_cache):,}")
print(f"Pares alcanzables: {n_alc:,}  ({n_alc/len(osm_dist_cache)*100:.1f}%)")

## 8. Construcción del grafo de sensores

Se usa `MultiDiGraph` porque entre el mismo par de sensores pueden existir
dos aristas independientes: `A→B` (ida) y `B→A` (vuelta), con distintos pesos.

**Reglas de conexión:**
- Todos los sensores se añaden siempre como nodos (nunca se elimina ninguno).
- Cada sensor se conecta con sus K vecinos más próximos **por carretera**.
- Si la distancia OSM no existe (ruta cortada), se usa haversine como fallback.

In [ ]:
G = nx.MultiDiGraph()

# ── Todos los sensores como nodos ─────────────────────────────────────────────
for _, row in sensors.iterrows():
    G.add_node(
        row["id"],
        sensor_id = str(row["id"]),
        nombre    = str(row["nombre"]),
        lat       = float(row["latitud"]),
        lon       = float(row["longitud"]),
        tipo_elem = str(row["tipo_elem"]),
        osm_node  = int(row["osm_node"]),
        color     = sensor_color.get(row["id"], COL_UNICO),
    )

# ── Conectar cada sensor con sus K vecinos válidos ────────────────────────────
cnt_osm = cnt_fallback = cnt_desc = 0

for i in range(len(sensors)):
    id_orig = sensors.iloc[i]["id"]
    osm_u   = int(sensors.iloc[i]["osm_node"])
    validos = 0

    for d_approx, j in zip(dist_m_approx[i][1:], idx_nn[i][1:]):
        if validos >= K_VECINOS:
            break
        if d_approx < MIN_DIST_M:
            cnt_desc += 1
            continue   # carril contrario → no conectar

        id_dest   = sensors.iloc[j]["id"]
        osm_v     = int(sensors.iloc[j]["osm_node"])
        dist_real = osm_dist_cache.get((osm_u, osm_v))

        if dist_real is not None:
            peso = dist_real
            src  = "osm"
            cnt_osm += 1
        else:
            peso = d_approx   # fallback: distancia haversine
            src  = "haversine"
            cnt_fallback += 1

        G.add_edge(id_orig, id_dest, weight=peso, length=peso, source=src)
        validos += 1

print(f"Aristas OSM real   : {cnt_osm:,}")
print(f"Aristas haversine  : {cnt_fallback:,}  (fallback: sin ruta OSM directa)")
print(f"Candidatos desc.   : {cnt_desc:,}  (< {MIN_DIST_M} m, carril contrario)")
print(f"\nGrafo construido:")
print(f"  Nodos   : {G.number_of_nodes()}")
print(f"  Aristas : {G.number_of_edges()}")

## 9. Garantizar conectividad máxima

Con K=3 vecinos dirigidos por sensor y la topología real de Madrid (ríos,
autovías, parques) pueden quedar componentes débilmente desconectadas.

Se añaden **aristas puente mínimas** entre la componente gigante y cada
componente aislada, en ambas direcciones, para no crear callejones sin salida.

In [ ]:
ids_all = sensors["id"].tolist()
id2idx  = {sid: i for i, sid in enumerate(ids_all)}
crd_g   = np.radians(sensors[["latitud", "longitud"]].values)
ids_arr = np.array(ids_all)


def _par_minimo_dist(set_a, set_b):
    """Par (u∈A, v∈B) de mínima distancia haversine entre dos conjuntos de sensores."""
    idx_a = [id2idx[n] for n in set_a]
    idx_b = [id2idx[n] for n in set_b]
    bt = BallTree(crd_g[idx_b], metric="haversine")
    dists, near = bt.query(crd_g[idx_a], k=1)
    k    = int(np.argmin(dists[:, 0]))
    dist = float(dists[k, 0] * R_EARTH)
    u    = ids_arr[idx_a[k]]
    v    = ids_arr[idx_b[int(near[k, 0])]]
    return u, v, dist


n_antes = nx.number_weakly_connected_components(G)
puentes = 0

while True:
    comps = list(nx.weakly_connected_components(G))
    if len(comps) == 1:
        break
    comps_ord  = sorted(comps, key=len, reverse=True)
    comp_gigan = comps_ord[0]
    for comp in comps_ord[1:]:
        u, v, d = _par_minimo_dist(comp, comp_gigan)
        G.add_edge(u, v, weight=d, length=d, source="bridge")
        G.add_edge(v, u, weight=d, length=d, source="bridge")
        comp_gigan = comp_gigan | comp
        puentes += 2

print(f"Componentes débiles antes : {n_antes}")
print(f"Puentes añadidos          : {puentes}")
print(f"Componentes débiles ahora : {nx.number_weakly_connected_components(G)}")

# ── Bidireccionalizar el grafo: garantizar conectividad fuerte ───────────────
# Las aristas K=3 de §8 son direccionales (A→B no implica B→A). Para que
# Dijkstra pueda encontrar ruta entre cualquier par de sensores, añadimos las
# aristas reversas que falten. §10 etiquetará bearing y sentido para todas.
n_rev = 0
for u, v, data in list(G.edges(data=True)):
    if not G.has_edge(v, u):
        G.add_edge(v, u,
                   weight=data["weight"],
                   length=data["length"],
                   source=data.get("source", "?") + "_rev")
        n_rev += 1

print(f"Aristas reversas añadidas : {n_rev}")
print(f"Aristas totales           : {G.number_of_edges():,}")
print(f"Componentes fuertes (SCC) : {nx.number_strongly_connected_components(G)}")

## 10. Etiquetado de sentidos de circulación

El **bearing** (rumbo compás) de cada arista indica la dirección de viaje:

| Rango | Dirección | Etiqueta |
|-------|-----------|----------|
| 0° – 179.9° | Norte / Este | `ida` |
| 180° – 359.9° | Sur / Oeste | `vuelta` |

Este criterio es universal y no depende del nombre del sensor.

In [ ]:
def compass_bearing(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Rumbo en grados [0, 360) desde (lat1, lon1) hacia (lat2, lon2)."""
    la1, la2 = math.radians(lat1), math.radians(lat2)
    dl = math.radians(lon2 - lon1)
    x  = math.sin(dl) * math.cos(la2)
    y  = math.cos(la1) * math.sin(la2) - math.sin(la1) * math.cos(la2) * math.cos(dl)
    return (math.degrees(math.atan2(x, y)) + 360) % 360


n_ida = n_vuelta = 0
for u, v, k in G.edges(keys=True):
    nu, nv = G.nodes[u], G.nodes[v]
    b = compass_bearing(nu["lat"], nu["lon"], nv["lat"], nv["lon"])
    sentido = "ida" if b < 180.0 else "vuelta"
    G[u][v][k]["sentido"] = sentido
    G[u][v][k]["bearing"] = round(b, 1)
    if sentido == "ida":
        n_ida += 1
    else:
        n_vuelta += 1

print(f"Aristas etiquetadas:")
print(f"  Ida    (0°–180°, N/E) : {n_ida:,}")
print(f"  Vuelta (180°–360°, S/O): {n_vuelta:,}")

## 11. Visualización estática — matplotlib

`connectionstyle='arc3,rad=0.08'` curva ligeramente las aristas para que
`A→B` e `B→A` sean visualmente distinguibles aunque coincidan en el mismo par.

- **Azul** → aristas de ida (bearing norte/este)
- **Rojo** → aristas de vuelta (bearing sur/oeste)
- **Puntos de colores** → color del sensor según sentido detectado

In [ ]:
fig, ax = plt.subplots(figsize=(16, 16))
ax.set_facecolor("#1a1a2e")
fig.patch.set_facecolor("#1a1a2e")

# Posiciones: (longitud, latitud) → orientación geográfica correcta
pos = {n: (d["lon"], d["lat"]) for n, d in G.nodes(data=True)}

# Aristas: vuelta primero (debajo), ida encima
for sentido, color, zorder in [("vuelta", "#C62828", 2), ("ida", "#1565C0", 3)]:
    elist = [
        (u, v)
        for u, v, d in G.edges(data=True)
        if d.get("sentido") == sentido
    ]
    nx.draw_networkx_edges(
        G, pos=pos, edgelist=elist,
        edge_color=color, alpha=0.55, width=0.9,
        arrows=True, arrowsize=8,
        connectionstyle="arc3,rad=0.08",
        ax=ax, node_size=0,
        min_source_margin=3, min_target_margin=3,
    )

# Nodos coloreados por sentido (azul / rojo / naranja)
node_colors = [G.nodes[n].get("color", COL_UNICO) for n in G.nodes()]
nx.draw_networkx_nodes(
    G, pos=pos, node_size=15,
    node_color=node_colors, alpha=0.92, ax=ax,
)

legend = [
    mpatches.Patch(color="#1565C0", label="Arista — ida (N/E)"),
    mpatches.Patch(color="#C62828", label="Arista — vuelta (S/O)"),
    mpatches.Patch(color=COL_A,     label="Sensor — sentido A"),
    mpatches.Patch(color=COL_B,     label="Sensor — sentido B"),
    mpatches.Patch(color=COL_UNICO, label="Sensor — sentido único"),
]
ax.legend(
    handles=legend, loc="lower right",
    facecolor="#2d2d44", edgecolor="white",
    labelcolor="white", fontsize=11,
)

ax.set_title(
    f"Red de sensores de tráfico — Madrid\n"
    f"{G.number_of_nodes()} sensores  ·  {G.number_of_edges()} aristas  ·  pesos: distancia OSM real (m)",
    color="white", fontsize=13, pad=14,
)
ax.axis("off")
plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "grafo_sensores_estatico.png",
    dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor(),
)
plt.show()
print(f"Figura guardada: {OUTPUT_DIR}/grafo_sensores_estatico.png")

## 12. Visualización interactiva — Folium

Mapa interactivo sobre **CartoDB Positron** (fondo claro que resalta los colores).

- **Azul** → sensor sentido A
- **Rojo** → sensor sentido B
- **Naranja** → sensor de sentido único
- **Triángulo ▲** rotado según el bearing → dirección del flujo en cada arista
- Cada nodo tiene un **popup** con id, nombre, coordenadas, tipo y grado

In [ ]:
lat_c = sensors["latitud"].mean()
lon_c = sensors["longitud"].mean()

m = folium.Map(location=[lat_c, lon_c], zoom_start=13, tiles="CartoDB positron")

# ── Aristas ───────────────────────────────────────────────────────────────────
for u, v, data in G.edges(data=True):
    nu, nv = G.nodes[u], G.nodes[v]
    p1 = (nu["lat"], nu["lon"])
    p2 = (nv["lat"], nv["lon"])

    folium.PolyLine(
        [p1, p2],
        weight=1.2, color=COL_EDGE, opacity=0.45,
        tooltip=(
            f"{u} → {v}  |  "
            f"{data.get('weight', 0):.0f} m  |"
            f"  {data.get('source', '?')}"
        ),
    ).add_to(m)

    # Flecha en el 65% del tramo indicando la dirección del flujo
    mid_lat = p1[0] + 0.65 * (p2[0] - p1[0])
    mid_lon = p1[1] + 0.65 * (p2[1] - p1[1])
    brng    = compass_bearing(*p1, *p2)
    folium.Marker(
        location=(mid_lat, mid_lon),
        icon=folium.DivIcon(
            html=(
                f'<div style="transform:rotate({brng}deg);'
                f'font-size:7px;color:{COL_ARROW};line-height:1;">&#9650;</div>'
            ),
            icon_size=(8, 8),
            icon_anchor=(4, 4),
        ),
    ).add_to(m)

# ── Nodos ─────────────────────────────────────────────────────────────────────
for node, data in G.nodes(data=True):
    color = data.get("color", COL_UNICO)
    folium.CircleMarker(
        location=[data["lat"], data["lon"]],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.9,
        popup=folium.Popup(
            f"<b>{node}</b><br>"
            f"{data['nombre']}<br>"
            f"({data['lat']:.5f}, {data['lon']:.5f})<br>"
            f"Tipo: {data['tipo_elem']}<br>"
            f"Salidas: {G.out_degree(node)}  Entradas: {G.in_degree(node)}",
            max_width=260,
        ),
    ).add_to(m)

# ── Leyenda HTML ──────────────────────────────────────────────────────────────
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:12px 16px;border-radius:8px;
     border:1px solid #ccc;font-size:13px;line-height:1.8;">
  <b>Sensores de tr&aacute;fico &mdash; Madrid</b><br>
  <span style="color:#1565C0;font-size:16px;">&#9679;</span> Sentido A<br>
  <span style="color:#B71C1C;font-size:16px;">&#9679;</span> Sentido B<br>
  <span style="color:#E65100;font-size:16px;">&#9679;</span> Sentido &uacute;nico<br>
  <span style="color:#37474F;">&#9650;</span> Direcci&oacute;n del flujo
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

out_html = OUTPUT_DIR / "grafo_sensores.html"
m.save(str(out_html))
print(f"Mapa interactivo guardado: {out_html}")
m

## 13. Análisis de conectividad

Métricas clave del grafo para verificar que la topología es correcta:

- **WCC** (componentes débiles): debería ser 1 tras los puentes del §7.
- **SCC** (componentes fuertes): cuántos nodos son mutuamente alcanzables.
- **Fuentes de aristas**: proporción de distancias reales OSM vs fallback.

In [ ]:
wcc = list(nx.weakly_connected_components(G))
scc = list(nx.strongly_connected_components(G))
gcc = max(scc, key=len)

grados = [G.in_degree(n) + G.out_degree(n) for n in G.nodes()]

weights = [d.get("weight", 0) for _, _, d in G.edges(data=True)]

print("=" * 56)
print("  RESUMEN DEL GRAFO")
print("=" * 56)
print(f"  Sensores (nodos)            : {G.number_of_nodes():,}")
print(f"  Aristas                     : {G.number_of_edges():,}")
print(f"  Componentes débiles  (WCC)  : {len(wcc)}")
print(f"  Componentes fuertes  (SCC)  : {len(scc)}")
print(f"  Tamaño SCC gigante          : {len(gcc):,}  ({len(gcc)/G.number_of_nodes()*100:.1f}%)")
print(f"  Grado medio (in+out)        : {np.mean(grados):.2f}")
print(f"  Grado mínimo                : {np.min(grados)}")
print(f"  Grado máximo                : {np.max(grados)}")
print(f"  Densidad                    : {nx.density(G):.6f}")
print(f"  Peso medio (distancia)      : {np.mean(weights):,.0f} m")
print(f"  Peso máximo                 : {np.max(weights):,.0f} m")
print(f"  Peso mínimo                 : {np.min(weights):,.0f} m")

cnt_src = {}
for _, _, d in G.edges(data=True):
    src = d.get("source", "?")
    cnt_src[src] = cnt_src.get(src, 0) + 1

print(f"\n  Fuentes de aristas:")
for src, cnt in sorted(cnt_src.items(), key=lambda x: -x[1]):
    pct = cnt / G.number_of_edges() * 100
    print(f"    {src:<12s}: {cnt:,}  ({pct:.1f}%)")
print("=" * 56)

## 14. Exportación

| Archivo | Formato | Uso |
|---------|---------|-----|
| `grafo_sensores.graphml` | GraphML | Importable en Gephi, QGIS, NetworkX |
| `sensores_metadata.csv` | CSV | Tabla de sensores con coordenadas y tipo |
| `grafo_sensores_estatico.png` | PNG | Figura estática para documentos |
| `grafo_sensores.html` | HTML | Mapa interactivo Folium |

El archivo GraphML contiene todos los atributos de nodos y aristas
(sensor_id, nombre, lat, lon, tipo_elem, weight, source, sentido, bearing).

In [ ]:
nx.write_graphml(G, OUTPUT_DIR / "grafo_sensores.graphml")
sensors.to_csv(OUTPUT_DIR / "sensores_metadata.csv", index=False)

print("Archivos exportados:")
for f in [
    OUTPUT_DIR / "grafo_sensores.graphml",
    OUTPUT_DIR / "sensores_metadata.csv",
    OUTPUT_DIR / "grafo_sensores_estatico.png",
    OUTPUT_DIR / "grafo_sensores.html",
]:
    size = f"{f.stat().st_size / 1024:.1f} KB" if f.exists() else "no generado"
    print(f"  {f.name:<35s}  {size}")

print(f"\nResumen final:")
print(f"  {G.number_of_nodes()} sensores  |  {G.number_of_edges()} aristas  |  "
      f"{nx.number_weakly_connected_components(G)} componente(s)")

## 15. Búsqueda de rutas entre sensores — Dijkstra

La función `find_route()` calcula la **ruta óptima** (mínima distancia total
por carretera) entre dos sensores usando el algoritmo de Dijkstra de NetworkX.

### Entradas
- `sensor_origen`  — `id` del sensor de partida (campo `id` del CSV)
- `sensor_destino` — `id` del sensor de llegada

### Salidas

| Campo | Tipo | Descripción |
|-------|------|-------------|
| `ruta` | `list` | Secuencia de `sensor_id` en orden de recorrido |
| `tramos` | `list[dict]` | Tramo a tramo: origen, destino, dist_m, sentido, fuente |
| `distancia_total` | `float` | Distancia total en metros |
| `n_sensores` | `int` | Número de sensores en la ruta (origen + intermedios + destino) |

El peso de las aristas es la **distancia real por carretera** (Dijkstra OSM)
o haversine como fallback cuando no existe ruta OSM directa (§7–§8).

> Se seleccionan automáticamente dos sensores dentro de la **componente fuerte
> gigante** (SCC) para garantizar que existe ruta dirigida entre ellos.
> Cambia `sensor_a` / `sensor_b` por cualquier par de la lista de sensores.

In [ ]:
def find_route(G, sensor_origen, sensor_destino):
    """
    Ruta óptima (mínima distancia) entre dos sensores.

    Parameters
    ----------
    G                : nx.MultiDiGraph  — grafo de sensores construido en §8
    sensor_origen    : sensor_id del punto de partida
    sensor_destino   : sensor_id del punto de llegada

    Returns
    -------
    dict con:
        ruta            — lista de sensor_id en orden de recorrido
        tramos          — list[dict]: orden, origen, destino, dist_m, sentido, source
        distancia_total — float en metros
        n_sensores      — int (incluye origen y destino)
    """
    if sensor_origen not in G.nodes:
        raise ValueError(f"Sensor origen '{sensor_origen}' no está en el grafo.")
    if sensor_destino not in G.nodes:
        raise ValueError(f"Sensor destino '{sensor_destino}' no está en el grafo.")

    try:
        ruta = nx.shortest_path(G, sensor_origen, sensor_destino, weight="weight")
    except nx.NetworkXNoPath:
        raise ValueError(
            f"No existe ruta dirigida entre '{sensor_origen}' y '{sensor_destino}'."
        )

    tramos, dist_total = [], 0.0
    for orden, (u, v) in enumerate(zip(ruta[:-1], ruta[1:]), start=1):
        edge_data = min(G[u][v].values(), key=lambda d: d.get("weight", float("inf")))
        dist = float(edge_data.get("weight", 0.0))
        tramos.append({
            "orden":   orden,
            "origen":  u,
            "destino": v,
            "dist_m":  round(dist, 1),
            "sentido": edge_data.get("sentido", "?"),
            "source":  edge_data.get("source",  "?"),
        })
        dist_total += dist

    return {
        "ruta":            ruta,
        "tramos":          tramos,
        "distancia_total": round(dist_total, 1),
        "n_sensores":      len(ruta),
    }


# ── Seleccionar dos sensores en la componente fuerte gigante (SCC) ────────
# La SCC garantiza que existe ruta dirigida en ambos sentidos.
scc_gigante = max(nx.strongly_connected_components(G), key=len)
scc_sorted  = sorted(scc_gigante)
sensor_a = scc_sorted[0]                   # primero alfabéticamente
sensor_b = scc_sorted[len(scc_sorted) // 2]  # sensor del centro

resultado = find_route(G, sensor_a, sensor_b)

# ── Mostrar resultados ───────────────────────────────────────────────────
print("=" * 64)
print("  RUTA ÓPTIMA (DIJKSTRA)")
print("=" * 64)
print(f"  Origen           : {sensor_a}")
print(f"  Destino          : {sensor_b}")
print(f"  Sensores en ruta : {resultado['n_sensores']}")
print(f"  Distancia total  : {resultado['distancia_total']:,.1f} m  "
      f"({resultado['distancia_total'] / 1000:.3f} km)")
print()
print(f"  {'#':>3}  {'Origen':<12}  {'Destino':<12}  {'Dist (m)':>9}  Sentido  Fuente")
print("  " + "-" * 62)
for t in resultado["tramos"]:
    print(
        f"  {t['orden']:>3}  {str(t['origen']):<12}  {str(t['destino']):<12}"
        f"  {t['dist_m']:>9,.1f}  {t['sentido']:<8} {t['source']}"
    )
print("  " + "-" * 62)
print(f"  {'TOT':>3}  {'':12}  {'':12}  {resultado['distancia_total']:>9,.1f}")
print("=" * 64)
print()
print(f"Componente fuerte gigante : {len(scc_gigante)} sensores")
print(f"Sensores disponibles      : {G.number_of_nodes()}")

# DataFrame de tramos para análisis
df_ruta = pd.DataFrame(resultado["tramos"])
display(df_ruta)

In [ ]:
# ── Matplotlib: ruta óptima resaltada sobre el grafo completo ───────────
ruta_ids   = resultado["ruta"]
ruta_set   = set(ruta_ids)
ruta_edges = list(zip(ruta_ids[:-1], ruta_ids[1:]))
nodos_inter = [n for n in ruta_ids if n != sensor_a and n != sensor_b]

fig, ax = plt.subplots(figsize=(15, 15))
ax.set_facecolor("#1a1a2e")
fig.patch.set_facecolor("#1a1a2e")

pos = {n: (d["lon"], d["lat"]) for n, d in G.nodes(data=True)}

# Todos los nodos del grafo (tenues)
nx.draw_networkx_nodes(
    G, pos=pos, ax=ax,
    node_size=6, node_color="#546E7A", alpha=0.30,
)

# Todas las aristas del grafo (muy tenues)
nx.draw_networkx_edges(
    G, pos=pos, ax=ax,
    edge_color="#37474F", alpha=0.15, width=0.4,
    arrows=False, node_size=0,
)

# Aristas de la ruta (verdes brillantes)
nx.draw_networkx_edges(
    G, pos=pos, ax=ax, edgelist=ruta_edges,
    edge_color="#76FF03", width=2.8, alpha=0.95,
    arrows=True, arrowsize=15, node_size=60,
    connectionstyle="arc3,rad=0.06",
    min_source_margin=5, min_target_margin=5,
)

# Sensores intermedios de la ruta (cyan)
if nodos_inter:
    nx.draw_networkx_nodes(
        G, pos=pos, ax=ax,
        nodelist=nodos_inter,
        node_size=55, node_color="#00E5FF", alpha=0.92,
    )

# Origen (verde) y Destino (rojo)
nx.draw_networkx_nodes(
    G, pos=pos, ax=ax, nodelist=[sensor_a],
    node_size=220, node_color="#00E676", alpha=1.0,
)
nx.draw_networkx_nodes(
    G, pos=pos, ax=ax, nodelist=[sensor_b],
    node_size=220, node_color="#FF1744", alpha=1.0,
)

# Anotaciones de origen y destino
_ox, _oy = pos[sensor_a]
_dx, _dy = pos[sensor_b]
ax.annotate(
    f"Origen\n{sensor_a}",
    xy=(_ox, _oy),
    xytext=(_ox + 0.003, _oy + 0.002),
    color="#00E676", fontsize=9, fontweight="bold", zorder=7,
    arrowprops=dict(arrowstyle="-", color="#00E676", lw=0.8),
)
ax.annotate(
    f"Destino\n{sensor_b}",
    xy=(_dx, _dy),
    xytext=(_dx + 0.003, _dy + 0.002),
    color="#FF1744", fontsize=9, fontweight="bold", zorder=7,
    arrowprops=dict(arrowstyle="-", color="#FF1744", lw=0.8),
)

legend_patches = [
    mpatches.Patch(color="#00E676", label=f"Origen  ({sensor_a})"),
    mpatches.Patch(color="#FF1744", label=f"Destino ({sensor_b})"),
    mpatches.Patch(color="#00E5FF", label=f"Intermedios ({len(nodos_inter)} sensores)"),
    mpatches.Patch(color="#76FF03", label=f"Ruta ({resultado['distancia_total']/1000:.2f} km)"),
    mpatches.Patch(color="#546E7A", label="Resto del grafo"),
]
ax.legend(handles=legend_patches, loc="lower right",
          facecolor="#2d2d44", edgecolor="white", labelcolor="white", fontsize=11)

ax.set_title(
    f"Ruta óptima (Dijkstra)  |  "
    f"{resultado['n_sensores']} sensores  ·  "
    f"{resultado['distancia_total']:,.0f} m  ({resultado['distancia_total']/1000:.2f} km)",
    color="white", fontsize=13, pad=14,
)
ax.axis("off")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "ruta_dijkstra.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Figura guardada: {OUTPUT_DIR}/ruta_dijkstra.png")

In [ ]:
# ── Folium: ruta óptima interactiva sobre el mapa real ───────────────────
lats_r  = [G.nodes[n]["lat"] for n in ruta_ids]
lons_r  = [G.nodes[n]["lon"] for n in ruta_ids]
lat_c_r = (min(lats_r) + max(lats_r)) / 2
lon_c_r = (min(lons_r) + max(lons_r)) / 2

m_ruta = folium.Map(
    location=[lat_c_r, lon_c_r], zoom_start=13, tiles="CartoDB positron"
)

# Resto de sensores del grafo (tenues, sin ruta)
for node, nd in G.nodes(data=True):
    if node in ruta_set:
        continue
    folium.CircleMarker(
        location=[nd["lat"], nd["lon"]],
        radius=3, color="#B0BEC5",
        fill=True, fill_color="#B0BEC5", fill_opacity=0.30,
        weight=0.5,
    ).add_to(m_ruta)

# Aristas de la ruta con flechas de dirección
for t in resultado["tramos"]:
    nu = G.nodes[t["origen"]]
    nv = G.nodes[t["destino"]]
    p1 = (nu["lat"], nu["lon"])
    p2 = (nv["lat"], nv["lon"])
    folium.PolyLine(
        [p1, p2], color="#76FF03", weight=5, opacity=0.90,
        tooltip=(
            f"Tramo {t['orden']}: {t['origen']} \u2192 {t['destino']}<br>"
            f"Distancia: {t['dist_m']:.0f} m  |  Sentido: {t['sentido']}"
        ),
    ).add_to(m_ruta)

    # Flecha en el punto medio
    mlat = p1[0] + 0.5 * (p2[0] - p1[0])
    mlon = p1[1] + 0.5 * (p2[1] - p1[1])
    brng = compass_bearing(p1[0], p1[1], p2[0], p2[1])
    folium.Marker(
        location=(mlat, mlon),
        icon=folium.DivIcon(
            html=(
                f'<div style="transform:rotate({brng}deg);'
                f'font-size:11px;color:#76FF03;">&#9650;</div>'
            ),
            icon_size=(12, 12), icon_anchor=(6, 6),
        ),
    ).add_to(m_ruta)

# Sensores intermedios
for i, nid in enumerate(ruta_ids[1:-1], start=1):
    nd = G.nodes[nid]
    folium.CircleMarker(
        location=[nd["lat"], nd["lon"]],
        radius=7, color="#00ACC1",
        fill=True, fill_color="#00E5FF", fill_opacity=0.90,
        tooltip=f"#{i + 1}  sensor_id: {nid}",
        popup=folium.Popup(
            (
                f"<b>sensor_id: {nid}</b><br>"
                f"{nd['nombre']}<br>"
                f"Tipo: {nd['tipo_elem']}<br>"
                f"Paso #{i + 1} de la ruta"
            ),
            max_width=230,
        ),
    ).add_to(m_ruta)

# Origen
nd_a = G.nodes[sensor_a]
folium.CircleMarker(
    location=[nd_a["lat"], nd_a["lon"]],
    radius=13, color="#00C853",
    fill=True, fill_color="#00E676", fill_opacity=1.0,
    tooltip=f"ORIGEN  sensor_id: {sensor_a}",
    popup=folium.Popup(
        (
            f"<b>ORIGEN</b><br>"
            f"sensor_id: {sensor_a}<br>"
            f"{nd_a['nombre']}<br>"
            f"Tipo: {nd_a['tipo_elem']}"
        ),
        max_width=230,
    ),
).add_to(m_ruta)

# Destino
nd_b = G.nodes[sensor_b]
folium.CircleMarker(
    location=[nd_b["lat"], nd_b["lon"]],
    radius=13, color="#D50000",
    fill=True, fill_color="#FF1744", fill_opacity=1.0,
    tooltip=f"DESTINO  sensor_id: {sensor_b}",
    popup=folium.Popup(
        (
            f"<b>DESTINO</b><br>"
            f"sensor_id: {sensor_b}<br>"
            f"{nd_b['nombre']}<br>"
            f"Tipo: {nd_b['tipo_elem']}"
        ),
        max_width=230,
    ),
).add_to(m_ruta)

# Leyenda
_leg = (
    "<div style='position:fixed;bottom:30px;left:30px;z-index:1000;"
    "background:white;padding:12px 16px;border-radius:8px;"
    "border:1px solid #ccc;font-size:13px;line-height:1.9;'>"
    "<b>Ruta &oacute;ptima &mdash; Dijkstra</b><br>"
    "<span style='color:#00E676;font-size:18px;'>&#9679;</span> Origen<br>"
    "<span style='color:#FF1744;font-size:18px;'>&#9679;</span> Destino<br>"
    "<span style='color:#00E5FF;font-size:15px;'>&#9679;</span> Sensores intermedios<br>"
    "<span style='color:#76FF03;font-size:13px;'>&#9644;</span> Ruta &oacute;ptima<br>"
    "<span style='color:#B0BEC5;font-size:13px;'>&#9679;</span> Resto del grafo<br>"
    f"<small>Distancia total: {resultado['distancia_total']:,.0f} m  ({resultado['distancia_total']/1000:.2f} km)"
    f" &middot; {resultado['n_sensores']} sensores</small>"
    "</div>"
)
m_ruta.get_root().html.add_child(folium.Element(_leg))

out_ruta = OUTPUT_DIR / "ruta_dijkstra.html"
m_ruta.save(str(out_ruta))
print(f"Mapa interactivo guardado: {out_ruta}")
m_ruta

## 16. Búsqueda de rutas por coordenadas GPS

En lugar de conocer el `sensor_id`, el usuario indica directamente las
**coordenadas geográficas** de origen y destino. El sistema:

1. **Snap lat/lon → sensor** — BallTree haversine encuentra el sensor más cercano.
2. **Dijkstra** — calcula la ruta óptima entre los dos sensores snap (§15).

### ¿Por qué BallTree haversine?

El **BallTree** organiza los puntos en un árbol de partición espacial esférica.
Una consulta de vecino más cercano cuesta **O(log n)** en lugar de **O(n)**
de la fuerza bruta (calcular distancia a todos los sensores).

Con n = 1 069 sensores: fuerza bruta ≈ 1 069 operaciones · BallTree ≈ 10.

La métrica usada es la **fórmula haversine**, distancia geodésica sobre la esfera:

$$d = 2R\,\arcsin\!\left(\sqrt{\sin^2\frac{\Delta\varphi}{2}
+ \cos\varphi_1\cos\varphi_2\sin^2\frac{\Delta\lambda}{2}}\right)$$

donde $R = 6\,371\,000$ m, $\varphi$ = latitud (rad), $\lambda$ = longitud (rad).

### Pipeline completo

```
(lat_orig, lon_orig) ──► BallTree ──► sensor_origen ──┐
                                                        ├──► Dijkstra ──► ruta
(lat_dest, lon_dest) ──► BallTree ──► sensor_destino ──┘
```

> El BallTree `tree` ya está construido en §7 sobre las coordenadas de todos
> los sensores. `find_nearest_sensor` lo reutiliza directamente.


In [ ]:
# ── Reutiliza: tree (BallTree de §7), sensors (DataFrame), G (grafo), R_EARTH ─

def find_nearest_sensor_candidates(lat: float, lon: float, k: int = 10):
    # K sensores más cercanos como candidatos para el snap.
    # Devuelve list[(row: pd.Series, dist_m: float)] ordenado por distancia.
    q = np.radians([[lat, lon]])
    dist_rad, idx = tree.query(q, k=k)
    out = []
    for i in range(k):
        row = sensors.iloc[int(idx[0, i])]
        d_m = float(dist_rad[0, i]) * R_EARTH
        out.append((row, round(d_m, 1)))
    return out


def find_nearest_sensor(lat: float, lon: float):
    # Sensor más cercano. Devuelve (sensor_id, dist_m, row).
    row, d_m = find_nearest_sensor_candidates(lat, lon, k=1)[0]
    return row["id"], d_m, row


def route_by_coords(lat_orig: float, lon_orig: float,
                    lat_dest: float, lon_dest: float,
                    k_fallback: int = 10):
    # Pipeline: snap lat/lon → sensor más cercano → Dijkstra.
    # Si no hay ruta dirigida entre los snaps más cercanos (sensores en SCC
    # distintas), prueba los K vecinos más cercanos a cada extremo y elige el
    # primer par con ruta, ordenado por distancia total de snap.
    cands_a = find_nearest_sensor_candidates(lat_orig, lon_orig, k=k_fallback)
    cands_b = find_nearest_sensor_candidates(lat_dest, lon_dest, k=k_fallback)

    pairs = []
    for row_a, d_a in cands_a:
        for row_b, d_b in cands_b:
            if row_a["id"] != row_b["id"]:
                pairs.append((row_a, row_b, d_a, d_b))
    pairs.sort(key=lambda p: p[2] + p[3])

    for row_a, row_b, d_a, d_b in pairs:
        sid_a = row_a["id"]
        sid_b = row_b["id"]
        if not nx.has_path(G, sid_a, sid_b):
            continue
        res = find_route(G, sid_a, sid_b)
        res["coord_origen"]   = (lat_orig, lon_orig)
        res["coord_destino"]  = (lat_dest, lon_dest)
        res["sensor_origen"]  = {
            "id": sid_a, "nombre": str(row_a["nombre"]),
            "lat": float(row_a["latitud"]), "lon": float(row_a["longitud"]),
        }
        res["sensor_destino"] = {
            "id": sid_b, "nombre": str(row_b["nombre"]),
            "lat": float(row_b["latitud"]), "lon": float(row_b["longitud"]),
        }
        res["snap_origen_m"]  = d_a
        res["snap_destino_m"] = d_b
        # Aviso si no se usaron los más cercanos
        if (row_a["id"] != cands_a[0][0]["id"] or
            row_b["id"] != cands_b[0][0]["id"]):
            print(f"Aviso: el par más cercano (sensores en SCC distintas) no "
                  f"tiene ruta dirigida. Se usa un par alternativo "
                  f"(snap +{d_a + d_b - cands_a[0][1] - cands_b[0][1]:.0f} m).")
        return res

    print(f"No se encontró ruta dirigida entre los {k_fallback} sensores más "
          f"cercanos al origen y destino. Aumenta k_fallback o revisa la "
          f"conectividad fuerte del grafo.")
    return None


# ── Ejemplo — cambia estas coordenadas por cualquier punto del área de sensores
LAT_ORIG, LON_ORIG = 40.4200, -3.7025   # Gran Vía
LAT_DEST, LON_DEST = 40.4525, -3.6730   # Chamartín

res_coords = route_by_coords(LAT_ORIG, LON_ORIG, LAT_DEST, LON_DEST)

if res_coords:
    so       = res_coords["sensor_origen"]
    sd       = res_coords["sensor_destino"]
    snap_a   = res_coords["snap_origen_m"]
    snap_b   = res_coords["snap_destino_m"]
    n_sens   = res_coords["n_sensores"]
    dist_tot = res_coords["distancia_total"]

    print("=" * 68)
    print("  RUTA POR COORDENADAS GPS")
    print("=" * 68)
    print(f"  Coord. origen    : ({LAT_ORIG}, {LON_ORIG})")
    print(f"  Sensor snap orig : {so['id']}  —  {so['nombre']}")
    print(f"  Distancia snap   : {snap_a:.1f} m")
    print()
    print(f"  Coord. destino   : ({LAT_DEST}, {LON_DEST})")
    print(f"  Sensor snap dest : {sd['id']}  —  {sd['nombre']}")
    print(f"  Distancia snap   : {snap_b:.1f} m")
    print()
    print(f"  Sensores en ruta : {n_sens}")
    print(f"  Distancia total  : {dist_tot:,.0f} m  ({dist_tot / 1000:.2f} km)")
    print("=" * 68)
    df_rc = pd.DataFrame(res_coords["tramos"])
    display(df_rc)


In [ ]:
# ── Matplotlib: coord GPS (★) ── snap (--) ── sensor ── ruta ─────────────────
if res_coords:
    so        = res_coords["sensor_origen"]
    sd        = res_coords["sensor_destino"]
    ruta_ids_c  = res_coords["ruta"]
    ruta_set_c  = set(ruta_ids_c)
    ruta_edges_c = list(zip(ruta_ids_c[:-1], ruta_ids_c[1:]))
    inter_c   = [n for n in ruta_ids_c
                 if n != so["id"] and n != sd["id"]]
    snap_a    = res_coords["snap_origen_m"]
    snap_b    = res_coords["snap_destino_m"]
    dist_tot  = res_coords["distancia_total"]
    n_sens    = res_coords["n_sensores"]

    fig, ax = plt.subplots(figsize=(15, 15))
    ax.set_facecolor("#1a1a2e")
    fig.patch.set_facecolor("#1a1a2e")
    pos = {n: (d["lon"], d["lat"]) for n, d in G.nodes(data=True)}

    # Grafo completo (muy tenue)
    nx.draw_networkx_nodes(G, pos=pos, ax=ax,
                           node_size=5, node_color="#546E7A", alpha=0.22)
    nx.draw_networkx_edges(G, pos=pos, ax=ax, edge_color="#37474F",
                           alpha=0.12, width=0.35, arrows=False, node_size=0)

    # Ruta óptima (verde brillante)
    nx.draw_networkx_edges(G, pos=pos, ax=ax, edgelist=ruta_edges_c,
                           edge_color="#76FF03", width=2.8, alpha=0.95,
                           arrows=True, arrowsize=15, node_size=60,
                           connectionstyle="arc3,rad=0.06",
                           min_source_margin=5, min_target_margin=5)

    # Sensores intermedios (cyan)
    if inter_c:
        nx.draw_networkx_nodes(G, pos=pos, ax=ax, nodelist=inter_c,
                               node_size=45, node_color="#00E5FF",
                               alpha=0.90)

    # Sensores snap: origen verde / destino rojo
    nx.draw_networkx_nodes(G, pos=pos, ax=ax, nodelist=[so["id"]],
                           node_size=220, node_color="#00E676", alpha=1.0)
    nx.draw_networkx_nodes(G, pos=pos, ax=ax, nodelist=[sd["id"]],
                           node_size=220, node_color="#FF1744", alpha=1.0)

    # Líneas snap: coord → sensor (punteadas)
    ax.plot([LON_ORIG, so["lon"]], [LAT_ORIG, so["lat"]],
            color="#FFD600", lw=1.8, linestyle="--", zorder=7, alpha=0.90)
    ax.plot([LON_DEST, sd["lon"]], [LAT_DEST, sd["lat"]],
            color="#FF6F00", lw=1.8, linestyle="--", zorder=7, alpha=0.90)

    # Coordenadas GPS (estrella dorada/naranja)
    ax.scatter([LON_ORIG], [LAT_ORIG], marker="*", s=400,
               color="#FFD600", zorder=9, edgecolors="white", linewidths=0.8)
    ax.scatter([LON_DEST], [LAT_DEST], marker="*", s=400,
               color="#FF6F00", zorder=9, edgecolors="white", linewidths=0.8)

    patches = [
        mpatches.Patch(color="#FFD600",
                       label=f"Coord. origen ({LAT_ORIG}, {LON_ORIG})"),
        mpatches.Patch(color="#FF6F00",
                       label=f"Coord. destino ({LAT_DEST}, {LON_DEST})"),
        mpatches.Patch(color="#00E676",
                       label=f"Sensor snap origen — {so['id']}  ({snap_a:.0f} m)"),
        mpatches.Patch(color="#FF1744",
                       label=f"Sensor snap destino — {sd['id']}  ({snap_b:.0f} m)"),
        mpatches.Patch(color="#00E5FF",
                       label=f"Intermedios ({len(inter_c)} sensores)"),
        mpatches.Patch(color="#76FF03",
                       label=f"Ruta ({dist_tot / 1000:.2f} km)"),
    ]
    ax.legend(handles=patches, loc="lower right",
              facecolor="#2d2d44", edgecolor="white", labelcolor="white",
              fontsize=10)
    ax.set_title(
        f"Búsqueda por coordenadas GPS  |  snap: {snap_a:.0f} m + {snap_b:.0f} m"
        f"  |  ruta: {dist_tot / 1000:.2f} km  ·  {n_sens} sensores",
        color="white", fontsize=12, pad=14)
    ax.axis("off")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "ruta_por_coords.png", dpi=150,
                bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    print(f"Figura guardada: {OUTPUT_DIR}/ruta_por_coords.png")


In [ ]:
# ── Folium: búsqueda por coordenadas GPS interactiva ─────────────────────────
if res_coords:
    so         = res_coords["sensor_origen"]
    sd         = res_coords["sensor_destino"]
    ruta_ids_c = res_coords["ruta"]
    ruta_set_c = set(ruta_ids_c)
    snap_a     = res_coords["snap_origen_m"]
    snap_b     = res_coords["snap_destino_m"]
    dist_tot   = res_coords["distancia_total"]
    n_sens     = res_coords["n_sensores"]

    all_lats = [LAT_ORIG, LAT_DEST] + [G.nodes[n]["lat"] for n in ruta_ids_c]
    all_lons = [LON_ORIG, LON_DEST] + [G.nodes[n]["lon"] for n in ruta_ids_c]
    lat_c_gc = (min(all_lats) + max(all_lats)) / 2
    lon_c_gc = (min(all_lons) + max(all_lons)) / 2

    m_gc = folium.Map(location=[lat_c_gc, lon_c_gc], zoom_start=13,
                      tiles="CartoDB positron")

    # Resto del grafo (tenue)
    for node, nd in G.nodes(data=True):
        if node in ruta_set_c:
            continue
        folium.CircleMarker(
            location=[nd["lat"], nd["lon"]], radius=2, color="#B0BEC5",
            fill=True, fill_color="#B0BEC5", fill_opacity=0.25, weight=0.3,
        ).add_to(m_gc)

    # Aristas de la ruta con flechas
    for t in res_coords["tramos"]:
        nu = G.nodes[t["origen"]]
        nv = G.nodes[t["destino"]]
        p1 = (nu["lat"], nu["lon"])
        p2 = (nv["lat"], nv["lon"])
        folium.PolyLine(
            [p1, p2], color="#76FF03", weight=5, opacity=0.90,
            tooltip=(f"Tramo {t['orden']}: {t['origen']} \u2192 {t['destino']}<br>"
                     f"{t['dist_m']:.0f} m  |  {t['sentido']}"),
        ).add_to(m_gc)
        mlat = p1[0] + 0.5 * (p2[0] - p1[0])
        mlon = p1[1] + 0.5 * (p2[1] - p1[1])
        brng = compass_bearing(p1[0], p1[1], p2[0], p2[1])
        folium.Marker(
            location=(mlat, mlon),
            icon=folium.DivIcon(
                html=(f'<div style="transform:rotate({brng}deg);'
                      f'font-size:10px;color:#76FF03;">&#9650;</div>'),
                icon_size=(12, 12), icon_anchor=(6, 6),
            ),
        ).add_to(m_gc)

    # Sensores intermedios
    for i, nid in enumerate(ruta_ids_c[1:-1], start=1):
        nd = G.nodes[nid]
        folium.CircleMarker(
            location=[nd["lat"], nd["lon"]], radius=6, color="#00ACC1",
            fill=True, fill_color="#00E5FF", fill_opacity=0.90,
            tooltip=f"#{i + 1}  sensor_id: {nid}",
            popup=folium.Popup(
                f"<b>sensor_id: {nid}</b><br>{nd['nombre']}<br>"
                f"Paso #{i + 1} de la ruta", max_width=220),
        ).add_to(m_gc)

    # Líneas snap punteadas
    folium.PolyLine(
        [(LAT_ORIG, LON_ORIG), (so["lat"], so["lon"])],
        color="#FFD600", weight=2.5, opacity=0.90, dash_array="8",
        tooltip=f"Snap origen: {snap_a:.1f} m",
    ).add_to(m_gc)
    folium.PolyLine(
        [(LAT_DEST, LON_DEST), (sd["lat"], sd["lon"])],
        color="#FF6F00", weight=2.5, opacity=0.90, dash_array="8",
        tooltip=f"Snap destino: {snap_b:.1f} m",
    ).add_to(m_gc)

    # Sensores snap
    folium.CircleMarker(
        location=[so["lat"], so["lon"]], radius=11, color="#00C853",
        fill=True, fill_color="#00E676", fill_opacity=1.0,
        tooltip=f"SENSOR ORIGEN  {so['id']}",
        popup=folium.Popup(
            f"<b>Sensor snap origen</b><br>ID: {so['id']}<br>"
            f"{so['nombre']}<br>Snap: {snap_a:.1f} m", max_width=240),
    ).add_to(m_gc)
    folium.CircleMarker(
        location=[sd["lat"], sd["lon"]], radius=11, color="#D50000",
        fill=True, fill_color="#FF1744", fill_opacity=1.0,
        tooltip=f"SENSOR DESTINO  {sd['id']}",
        popup=folium.Popup(
            f"<b>Sensor snap destino</b><br>ID: {sd['id']}<br>"
            f"{sd['nombre']}<br>Snap: {snap_b:.1f} m", max_width=240),
    ).add_to(m_gc)

    # Coordenadas GPS (estrella con DivIcon)
    for lat_pt, lon_pt, color_pt, label_pt, snap_pt, sname in [
        (LAT_ORIG, LON_ORIG, "#FFD600", "ORIGEN",  snap_a, so["id"]),
        (LAT_DEST, LON_DEST, "#FF6F00", "DESTINO", snap_b, sd["id"]),
    ]:
        folium.Marker(
            location=[lat_pt, lon_pt],
            icon=folium.DivIcon(
                html=(f'<div style="font-size:24px;color:{color_pt};'
                      f'text-shadow:0 0 4px #000;line-height:1;">&#9733;</div>'),
                icon_size=(26, 26), icon_anchor=(13, 13),
            ),
            tooltip=f"Coord. {label_pt} ({lat_pt}, {lon_pt})",
            popup=folium.Popup(
                f"<b>Coordenada {label_pt}</b><br>"
                f"Lat: {lat_pt}<br>Lon: {lon_pt}<br>"
                f"Sensor snap: {sname}<br>Snap: {snap_pt:.1f} m",
                max_width=230),
        ).add_to(m_gc)

    # Leyenda
    _leg_gc = (
        "<div style='position:fixed;bottom:30px;left:30px;z-index:1000;"
        "background:white;padding:12px 16px;border-radius:8px;"
        "border:1px solid #ccc;font-size:13px;line-height:1.9;'>"
        "<b>B&uacute;squeda por coordenadas GPS</b><br>"
        "<span style='color:#FFD600;font-size:20px;'>&#9733;</span>"
        " Coord. origen<br>"
        "<span style='color:#FF6F00;font-size:20px;'>&#9733;</span>"
        " Coord. destino<br>"
        "<span style='color:#00E676;font-size:16px;'>&#9679;</span>"
        " Sensor snap origen<br>"
        "<span style='color:#FF1744;font-size:16px;'>&#9679;</span>"
        " Sensor snap destino<br>"
        "<span style='color:#00E5FF;font-size:13px;'>&#9679;</span>"
        " Sensores intermedios<br>"
        "<span style='color:#76FF03;'>&#9644;</span>"
        " Ruta &oacute;ptima<br>"
        "<span style='color:#FFD600;'>&#9148;</span>"
        " Snap origen &nbsp;"
        "<span style='color:#FF6F00;'>&#9148;</span>"
        " Snap destino<br>"
        f"<small>Ruta: {dist_tot / 1000:.2f} km &middot; {n_sens} sensores<br>"
        f"Snap: {snap_a:.0f} m + {snap_b:.0f} m</small>"
        "</div>"
    )
    m_gc.get_root().html.add_child(folium.Element(_leg_gc))

    out_gc = OUTPUT_DIR / "ruta_por_coords.html"
    m_gc.save(str(out_gc))
    print(f"Mapa interactivo guardado: {out_gc}")
    m_gc


## 17. Routing sobre la red vial OSM + identificación de sensores

### Arquitectura correcta para predicción de tráfico

En §15 y §16 el grafo de sensores actuaba como red de enrutamiento. El problema:
los sensores no están a lo largo de las calles, sino donde el ayuntamiento decidió
ponerlos. Por eso las rutas zigzaguean y son artificialmente largas.

**Aquí separamos correctamente las dos capas:**

| Capa | Rol |
|------|-----|
| **Red vial OSM** (`G_osm`) | Enrutamiento real por carretera |
| **Sensores** | Observaciones de tráfico a lo largo de la ruta |

### Pipeline

```
(lat_orig, lon_orig) ──► ox.nearest_nodes ──► osm_node_orig ──┐
                                                                ├──► Dijkstra OSM ──► ruta_osm
(lat_dest, lon_dest) ──► ox.nearest_nodes ──► osm_node_dest ──┘                       │
                                                                                       ▼
                                              sensores cuyo osm_node ∈ ruta_osm  ◄── filtro
```

### Por qué es lo correcto para el TFG

1. **La ruta es realista** — sigue calles reales con geometría OSM.
2. **Los sensores se identifican como observaciones**, ordenados por posición a lo
   largo de la ruta. Esa secuencia es lo que se le pasa al modelo de deep learning
   para que prediga tráfico/velocidad en cada uno.
3. **Separación de responsabilidades** — la red vial cambia rara vez, los sensores
   pueden añadirse o retirarse sin alterar el motor de enrutamiento.

> El map matching de §6 (`sensors["osm_node"]`) es la clave: ya tenemos la
> correspondencia sensor → nodo vial, así que filtrar sensores en la ruta es
> trivial.


In [ ]:
# ── Reutiliza: G_osm (red vial), sensors (con osm_node de §6) ────────────────

def route_by_coords_osm(lat_orig: float, lon_orig: float,
                        lat_dest: float, lon_dest: float):
    # Pipeline: snap a nodo OSM → Dijkstra sobre red vial → sensores en ruta.
    # Devuelve dict con:
    #   ruta_osm        — secuencia de nodos OSM (id de OpenStreetMap)
    #   geometry        — lista de (lat, lon) siguiendo la geometría real de calles
    #   distancia_total — metros recorridos por carretera
    #   sensores_en_ruta — DataFrame de sensores cuyo osm_node está en la ruta,
    #                     ordenados por posición a lo largo de la ruta
    # Subgrafo enrutable: mayor componente fuertemente conexo de G_osm.
    # G_osm se descargó con retain_all=True (conserva islas desconectadas)
    # para que la celda §6 pueda map-matchear sensores en calles pequeñas;
    # pero para Dijkstra origen→destino necesitamos un subgrafo donde TODO
    # par de nodos esté conectado por aristas dirigidas → SCC máximo.
    global _G_osm_routable
    if "_G_osm_routable" not in globals() or _G_osm_routable is None:
        scc = max(nx.strongly_connected_components(G_osm), key=len)
        _G_osm_routable = G_osm.subgraph(scc).copy()
        print(f"[routing] subgrafo enrutable: {len(_G_osm_routable.nodes):,} nodos / "
              f"{len(_G_osm_routable.edges):,} aristas (de {len(G_osm.nodes):,} totales)")

    # Snap a nodo OSM dentro del subgrafo enrutable (no a islas desconectadas).
    osm_orig = int(ox.nearest_nodes(_G_osm_routable, X=lon_orig, Y=lat_orig))
    osm_dest = int(ox.nearest_nodes(_G_osm_routable, X=lon_dest, Y=lat_dest))

    if osm_orig == osm_dest:
        print("Origen y destino se mapean al mismo nodo OSM. "
              "Introduce coordenadas más separadas.")
        return None

    try:
        ruta_osm = nx.shortest_path(_G_osm_routable, osm_orig, osm_dest, weight="length")
    except nx.NetworkXNoPath:
        print(f"No hay ruta OSM entre los nodos {osm_orig} y {osm_dest}.")
        return None

    # Reconstruir geometría real de la ruta + distancia total
    geometry  = []
    dist_total = 0.0
    for u, v in zip(ruta_osm[:-1], ruta_osm[1:]):
        edata_dict = _G_osm_routable.get_edge_data(u, v)
        if edata_dict is None:
            continue
        best = min(edata_dict.values(),
                   key=lambda d: d.get("length", float("inf")))
        dist_total += float(best.get("length", 0.0))
        if "geometry" in best and best["geometry"] is not None:
            # LineString en (lon, lat) — lo invertimos a (lat, lon) para Folium
            for lon, lat in best["geometry"].coords:
                geometry.append((float(lat), float(lon)))
        else:
            geometry.append((float(_G_osm_routable.nodes[u]["y"]), float(_G_osm_routable.nodes[u]["x"])))
            geometry.append((float(_G_osm_routable.nodes[v]["y"]), float(_G_osm_routable.nodes[v]["x"])))

    # Sensores cuyo osm_node está en la ruta, ordenados por posición
    pos_in_route = {n: i for i, n in enumerate(ruta_osm)}
    mask = sensors["osm_node"].isin(pos_in_route)
    sens_route = sensors[mask].copy()
    sens_route["pos_ruta"] = sens_route["osm_node"].map(pos_in_route)
    sens_route = sens_route.sort_values("pos_ruta").reset_index(drop=True)

    return {
        "coord_origen":      (lat_orig, lon_orig),
        "coord_destino":     (lat_dest, lon_dest),
        "osm_origen":        osm_orig,
        "osm_destino":       osm_dest,
        "ruta_osm":          ruta_osm,
        "geometry":          geometry,
        "distancia_total":   round(dist_total, 1),
        "sensores_en_ruta":  sens_route,
        "n_sensores":        len(sens_route),
        "n_nodos_osm":       len(ruta_osm),
    }


# ── Ejemplo ───────────────────────────────────────────────────────────────────
LAT_ORIG_OSM, LON_ORIG_OSM = 40.4200, -3.7025   # Gran Vía
LAT_DEST_OSM, LON_DEST_OSM = 40.4525, -3.6730   # Chamartín

res_osm = route_by_coords_osm(
    LAT_ORIG_OSM, LON_ORIG_OSM, LAT_DEST_OSM, LON_DEST_OSM
)

if res_osm:
    dist_tot   = res_osm["distancia_total"]
    n_sens     = res_osm["n_sensores"]
    n_nodos    = res_osm["n_nodos_osm"]

    print("=" * 70)
    print("  RUTA SOBRE RED VIAL OSM + SENSORES EN RUTA")
    print("=" * 70)
    print(f"  Coord. origen     : ({LAT_ORIG_OSM}, {LON_ORIG_OSM})")
    print(f"  Coord. destino    : ({LAT_DEST_OSM}, {LON_DEST_OSM})")
    print(f"  Nodo OSM origen   : {res_osm['osm_origen']}")
    print(f"  Nodo OSM destino  : {res_osm['osm_destino']}")
    print()
    print(f"  Nodos OSM en ruta : {n_nodos}")
    print(f"  Distancia total   : {dist_tot:,.0f} m  ({dist_tot / 1000:.2f} km)")
    print()
    print(f"  Sensores observados a lo largo de la ruta : {n_sens}")
    print("=" * 70)
    if n_sens:
        cols = ["pos_ruta", "id", "nombre", "tipo_elem", "osm_node"]
        display(res_osm["sensores_en_ruta"][cols])
    else:
        print("  La ruta no atraviesa ningún sensor (zona sin cobertura).")


In [ ]:
# ── Matplotlib: ruta OSM real + sensores observados ─────────────────────────
if res_osm:
    sens_route = res_osm["sensores_en_ruta"]
    geom       = res_osm["geometry"]
    dist_tot   = res_osm["distancia_total"]
    n_sens     = res_osm["n_sensores"]

    # Plot del grafo OSM con la ruta resaltada (función nativa de OSMnx)
    fig, ax = ox.plot_graph_route(
        G_osm, res_osm["ruta_osm"],
        route_color="#76FF03", route_linewidth=4, route_alpha=0.95,
        bgcolor="#1a1a2e",
        node_size=0,
        edge_color="#37474F", edge_linewidth=0.35, edge_alpha=0.45,
        figsize=(15, 15),
        show=False, close=False,
    )

    # Sensores observados a lo largo de la ruta (cyan)
    if not sens_route.empty:
        ax.scatter(sens_route["longitud"], sens_route["latitud"],
                   s=55, c="#00E5FF", alpha=0.95, zorder=5,
                   edgecolors="white", linewidths=0.6,
                   label=f"Sensores en ruta ({n_sens})")

    # Coordenadas GPS introducidas (estrellas)
    ax.scatter([LON_ORIG_OSM], [LAT_ORIG_OSM], marker="*", s=450,
               color="#FFD600", zorder=9,
               edgecolors="white", linewidths=0.8,
               label=f"Origen ({LAT_ORIG_OSM}, {LON_ORIG_OSM})")
    ax.scatter([LON_DEST_OSM], [LAT_DEST_OSM], marker="*", s=450,
               color="#FF6F00", zorder=9,
               edgecolors="white", linewidths=0.8,
               label=f"Destino ({LAT_DEST_OSM}, {LON_DEST_OSM})")

    ax.legend(loc="lower right", facecolor="#2d2d44", edgecolor="white",
              labelcolor="white", fontsize=10)
    ax.set_title(
        f"Ruta OSM real  |  {dist_tot:,.0f} m  ({dist_tot / 1000:.2f} km)  ·  "
        f"{n_sens} sensores observados",
        color="white", fontsize=12, pad=14,
    )
    fig.savefig(OUTPUT_DIR / "ruta_osm_real.png", dpi=150,
                bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    print(f"Figura guardada: {OUTPUT_DIR}/ruta_osm_real.png")


In [ ]:
# ── Folium: ruta OSM real interactiva con sensores observados ───────────────
if res_osm:
    sens_route = res_osm["sensores_en_ruta"]
    geom       = res_osm["geometry"]
    dist_tot   = res_osm["distancia_total"]
    n_sens     = res_osm["n_sensores"]

    lats_g = [p[0] for p in geom]
    lons_g = [p[1] for p in geom]
    lat_c  = (min(lats_g) + max(lats_g)) / 2
    lon_c  = (min(lons_g) + max(lons_g)) / 2

    m_osm = folium.Map(location=[lat_c, lon_c], zoom_start=14,
                       tiles="CartoDB positron")

    # Ruta real siguiendo la geometría de las calles
    folium.PolyLine(
        geom, color="#76FF03", weight=6, opacity=0.95,
        tooltip=f"Ruta OSM: {dist_tot:,.0f} m ({dist_tot / 1000:.2f} km)",
    ).add_to(m_osm)

    # Sensores en ruta (cyan, ordenados por posición)
    for _, row in sens_route.iterrows():
        folium.CircleMarker(
            location=[row["latitud"], row["longitud"]],
            radius=7, color="#00ACC1",
            fill=True, fill_color="#00E5FF", fill_opacity=0.95,
            tooltip=f"#{int(row['pos_ruta'])}  sensor_id: {row['id']}",
            popup=folium.Popup(
                f"<b>sensor_id: {row['id']}</b><br>"
                f"{row['nombre']}<br>"
                f"Tipo: {row['tipo_elem']}<br>"
                f"Posición en ruta: nodo OSM #{int(row['pos_ruta'])}<br>"
                f"osm_node: {int(row['osm_node'])}",
                max_width=240,
            ),
        ).add_to(m_osm)

    # Coordenadas GPS introducidas (estrellas)
    for lat_pt, lon_pt, color_pt, label_pt in [
        (LAT_ORIG_OSM, LON_ORIG_OSM, "#FFD600", "ORIGEN"),
        (LAT_DEST_OSM, LON_DEST_OSM, "#FF6F00", "DESTINO"),
    ]:
        folium.Marker(
            location=[lat_pt, lon_pt],
            icon=folium.DivIcon(
                html=(f'<div style="font-size:26px;color:{color_pt};'
                      f'text-shadow:0 0 4px #000;line-height:1;">&#9733;</div>'),
                icon_size=(28, 28), icon_anchor=(14, 14),
            ),
            tooltip=f"Coord. {label_pt} ({lat_pt}, {lon_pt})",
            popup=folium.Popup(
                f"<b>{label_pt}</b><br>Lat: {lat_pt}<br>Lon: {lon_pt}",
                max_width=200,
            ),
        ).add_to(m_osm)

    # Leyenda
    _leg = (
        "<div style='position:fixed;bottom:30px;left:30px;z-index:1000;"
        "background:white;padding:12px 16px;border-radius:8px;"
        "border:1px solid #ccc;font-size:13px;line-height:1.9;'>"
        "<b>Routing OSM real + sensores</b><br>"
        "<span style='color:#FFD600;font-size:20px;'>&#9733;</span> Origen GPS<br>"
        "<span style='color:#FF6F00;font-size:20px;'>&#9733;</span> Destino GPS<br>"
        "<span style='color:#76FF03;font-size:14px;'>&#9644;</span>"
        " Ruta real por carretera<br>"
        "<span style='color:#00E5FF;font-size:16px;'>&#9679;</span>"
        " Sensores observados ({n})<br>"
        f"<small>Distancia: {dist_tot:,.0f} m ({dist_tot / 1000:.2f} km)<br>"
        f"Sensores en ruta: {n_sens}</small>"
        "</div>"
    ).replace("({n})", f"({n_sens})")
    m_osm.get_root().html.add_child(folium.Element(_leg))

    out_osm = OUTPUT_DIR / "ruta_osm_real.html"
    m_osm.save(str(out_osm))
    print(f"Mapa interactivo guardado: {out_osm}")
    m_osm
